# 04 · Agente que usa a memória como ferramenta

Aqui o **modelo decide** quando consultar a memória, quando pesquisar nos documentos e quando
guardar um fato, chamando ferramentas (function calling do Ollama). Útil quando o agente já tem
outras ferramentas e você não quer buscar contexto a cada mensagem.

As ferramentas são funções Python comuns sobre a API do pi-memory; o mesmo desenho vale para
qualquer framework de agentes (o servidor MCP previsto expõe essas operações direto).

> **Instalação** (uma vez):
> ```bash
> pip install "pi-memory[local]"
> ollama pull qwen2.5:14b
> ```
> Todos os dados deste notebook são fictícios. As saídas abaixo vêm de uma execução real.

In [1]:
import json
import shutil
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")  # aviso do tqdm no Jupyter

def mostrar(obj, limite=1500):
    """JSON legível (o mesmo formato da biblioteca, do HTTP e do MCP)."""
    texto = json.dumps(obj, ensure_ascii=False, indent=2)
    print(texto if len(texto) <= limite else texto[:limite] + "\n…")

def pasta_limpa(nome):
    """Cada notebook começa do zero: apaga a pasta de dados da execução anterior."""
    shutil.rmtree(nome, ignore_errors=True)
    return nome

In [2]:
import httpx
from pi_memory import PiMemory
from pi_memory.llms import OllamaLLM

OLLAMA, MODELO, DONO = "http://localhost:11434", "qwen2.5:14b", "ana"
pm = PiMemory(path=pasta_limpa("dados_04"), llm=OllamaLLM(MODELO, timeout=180))
pm.documents.add(Path("../../eval/fixtures/vendas.xlsx"), owner=DONO)
pm.memories.remember("O usuário prefere respostas curtas", owner=DONO, topic="perfil")

{'id': '0a33e8d6-6d3b-40bf-91b3-287479e35550',
 'text': 'O usuário prefere respostas curtas',
 'owner': 'ana',
 'topic': 'perfil',
 'created_at': '2026-11-04T22:13:30.010902Z',
 'updated_at': '2026-11-04T22:13:30.010902Z',
 'metadata': {}}

## As ferramentas

In [3]:
def buscar_memorias(consulta: str) -> dict:
    r = pm.memories.search(consulta, owner=DONO, limit=3)["results"]
    return {"memorias": [m["text"] for m in r]}

def buscar_documentos(consulta: str) -> dict:
    r = pm.documents.search(consulta, owner=DONO, limit=3)["results"]
    return {"trechos": [{"fonte": t["citation"]["label"], "texto": t["text"]} for t in r]}

def guardar_fato(fato: str) -> dict:
    m = pm.memories.remember(fato, owner=DONO)
    return {"guardado": m["text"]}

FERRAMENTAS = {f.__name__: f for f in (buscar_memorias, buscar_documentos, guardar_fato)}

def esquema(nome, descricao, parametro):
    return {"type": "function", "function": {
        "name": nome, "description": descricao,
        "parameters": {"type": "object", "required": [parametro],
                       "properties": {parametro: {"type": "string"}}}}}

TOOLS = [
    esquema("buscar_memorias", "Busca o que já se sabe sobre o usuário (preferências, família, trabalho).", "consulta"),
    esquema("buscar_documentos", "Pesquisa nos documentos do usuário; devolve trechos com a fonte.", "consulta"),
    esquema("guardar_fato", "Guarda um fato duradouro sobre o usuário, escrito como 'O usuário ...'.", "fato"),
]

## O laço do agente

Enquanto o modelo pedir ferramentas, executa e devolve o resultado; quando ele responder com
texto, termina.

In [4]:
SISTEMA = ("Você é um assistente em português do Brasil. Você tem ferramentas e deve usá-las:\n"
           "- buscar_documentos: SEMPRE que a pergunta for sobre vendas, notas, clientes, contratos "
           "ou outros dados do usuário, antes de dizer que não sabe; cite a fonte;\n"
           "- buscar_memorias: quando precisar de algo que o usuário contou antes;\n"
           "- guardar_fato: quando o usuário contar um fato novo e duradouro sobre si.")

def agente(mensagem, historico):
    historico.append({"role": "user", "content": mensagem})
    print(f"🧑 {mensagem}")
    for _ in range(6):
        msg = httpx.post(f"{OLLAMA}/api/chat", timeout=180, json={
            "model": MODELO, "stream": False, "tools": TOOLS, "options": {"temperature": 0},
            "messages": [{"role": "system", "content": SISTEMA}, *historico]}).json()["message"]
        historico.append(msg)
        chamadas = msg.get("tool_calls") or []
        if not chamadas:
            print(f"🤖 {msg['content']}\n")
            return
        for chamada in chamadas:
            nome, args = chamada["function"]["name"], chamada["function"]["arguments"]
            resultado = FERRAMENTAS[nome](**args)
            print(f"   🔧 {nome}({json.dumps(args, ensure_ascii=False)}) → "
                  f"{json.dumps(resultado, ensure_ascii=False)[:160]}")
            historico.append({"role": "tool", "content": json.dumps(resultado, ensure_ascii=False)})

historico = []
agente("Oi! Comecei hoje num emprego novo: sou analista de dados na Livraria Exemplo.", historico)
agente("Quanto vendemos para a Padaria Imaginária e qual foi a nota?", historico)
agente("Você lembra onde eu trabalho e como prefiro as respostas?", []) # histórico vazio

🧑 Oi! Comecei hoje num emprego novo: sou analista de dados na Livraria Exemplo.


🤖 IÊ, parabéns pelo novo emprego! Como analista de dados na Livraria Exemplo, você deve estar ocupado analisando tendências e comportamentos dos clientes para ajudar a empresa a tomar decisões informadas. Se precisar de ajuda com algo relacionado ao seu trabalho, estou aqui para ajudar!

🧑 Quanto vendemos para a Padaria Imaginária e qual foi a nota?


   🔧 buscar_documentos({"consulta": "vendas para a Padaria Imaginária e notas"}) → {"trechos": [{"fonte": "vendas.xlsx, aba Vendas, linhas 1–5", "texto": "Nota: NF-4519 | Cliente: Loja Fictícia SA | Valor: 1200 Nota: NF-4520 | Cliente: Mercado


🤖 De acordo com os documentos, a Padaria Imaginária comprou um item de vocês por R$ 430, conforme a nota fiscal NF-4521. Não há outras transações listadas para essa cliente no documento fornecido.

🧑 Você lembra onde eu trabalho e como prefiro as respostas?


   🔧 buscar_memorias({"consulta": "trabalho e preferências de resposta"}) → {"memorias": ["O usuário prefere respostas curtas"]}


🤖 Você prefere respostas curtas. Quanto ao seu trabalho, não tenho informações específicas sobre onde você trabalha no momento. Se quiser, pode me contar mais sobre isso.



In [5]:
for m in pm.memories.list(owner=DONO)["memories"]:
    print(f"- [{m['topic']}] {m['text']}")

- [perfil] O usuário prefere respostas curtas


## O ponto fraco: guardar depende do modelo

Veja a primeira mensagem acima: o usuário contou do emprego novo e o modelo pode **não** chamar
`guardar_fato`; se isso acontece, na terceira pergunta (histórico vazio) o agente não sabe onde o
usuário trabalha. Buscar pela ferramenta funciona bem (ele consultou a planilha e citou a nota),
mas confiar ao modelo a decisão de **guardar** é frágil.

## Variante robusta: ferramentas para buscar, aprendizado automático

As ferramentas continuam para **buscar**; depois de cada troca, o seu código entrega a conversa
ao `memories.add`, que extrai e decide sozinho (como no notebook 03). Mesmas três mensagens,
memória limpa:

In [6]:
pm.memories.delete_all(owner=DONO)
pm.memories.remember("O usuário prefere respostas curtas", owner=DONO, topic="perfil")

def agente_que_aprende(mensagem, historico):
    agente(mensagem, historico)
    troca = [{"role": "user", "content": mensagem},
             {"role": "assistant", "content": historico[-1].get("content", "")}]
    for d in pm.memories.add(troca, owner=DONO)["decisions"]:
        print(f"   💾 {d['event']}: {d['text']}")
    print()

historico = []
agente_que_aprende("Oi! Comecei hoje num emprego novo: sou analista de dados na Livraria Exemplo.", historico)
agente_que_aprende("Quanto vendemos para a Padaria Imaginária e qual foi a nota?", historico)
agente_que_aprende("Você lembra onde eu trabalho e como prefiro as respostas?", [])

🧑 Oi! Comecei hoje num emprego novo: sou analista de dados na Livraria Exemplo.


🤖 IÊ, parabéns pelo novo emprego! Como analista de dados na Livraria Exemplo, você deve estar ocupado analisando tendências e comportamentos dos clientes para ajudar a empresa a tomar decisões informadas. Se precisar de ajuda com algo relacionado ao seu trabalho, estou aqui para ajudar!



   💾 add: O usuário começou um novo emprego hoje como analista de dados na Livraria Exemplo.

🧑 Quanto vendemos para a Padaria Imaginária e qual foi a nota?


   🔧 buscar_documentos({"consulta": "vendas para a Padaria Imaginária e notas"}) → {"trechos": [{"fonte": "vendas.xlsx, aba Vendas, linhas 1–5", "texto": "Nota: NF-4519 | Cliente: Loja Fictícia SA | Valor: 1200 Nota: NF-4520 | Cliente: Mercado


🤖 De acordo com os documentos, a Padaria Imaginária comprou produtos da Livraria Exemplo por um valor de R$ 430, conforme a nota fiscal NF-4521. Não há informações adicionais sobre a nota específica no documento fornecido.




🧑 Você lembra onde eu trabalho e como prefiro as respostas?


   🔧 buscar_memorias({"consulta": "trabalho e preferências de resposta"}) → {"memorias": ["O usuário começou um novo emprego hoje como analista de dados na Livraria Exemplo.", "O usuário prefere respostas curtas"]}


🤖 Você começou um novo emprego hoje como analista de dados na Livraria Exemplo e prefere receber respostas curtas.



   💾 none: O usuário trabalha como analista de dados na Livraria Exemplo
   💾 update: O usuário prefere receber respostas curtas



In [7]:
for m in pm.memories.list(owner=DONO)["memories"]:
    print(f"- [{m['topic']}] {m['text']}")
pm.close()

- [perfil] O usuário prefere receber respostas curtas
- [trabalho] O usuário começou um novo emprego hoje como analista de dados na Livraria Exemplo.


## Contexto × ferramenta: quando usar cada um

| | Memória como contexto (notebook 03) | Memória como ferramenta (este) |
|---|---|---|
| Quem decide buscar | o seu código, toda mensagem | o modelo |
| Previsibilidade | alta | depende do modelo seguir as ferramentas |
| Custo | 1 busca por mensagem | só quando o modelo pede |
| Guardar | `memories.add` com decisão completa | depende de o modelo chamar a ferramenta; a variante robusta usa `memories.add` automático |
| Bom para | assistentes pessoais | agentes com muitas ferramentas |